In [ ]:
class Vector:
    def __init__(self, components):
        self.components = list(components)
        self.dim = len(self.components)

    def __add__(self, other):
        return Vector([a + b for a, b in zip(self.components, other.components)])

    def __sub__(self, other):
        return Vector([a - b for a, b in zip(self.components, other.components)])

    def dot(self, other):
        return sum(a * b for a, b in zip(self.components, other.components))

    def magnitude(self):
        return sum(x**2 for x in self.components) ** 0.5

    def normalize(self):
        mag = self.magnitude()
        return Vector([x / mag for x in self.components])

    def cosine_similarity(self, other):
        return self.dot(other) / (self.magnitude() * other.magnitude())

    def __repr__(self):
        return f"Vector({self.components})"


a = Vector([1, 2, 3])
b = Vector([4, 5, 6])

print(f"a + b = {a + b}")
print(f"a · b = {a.dot(b)}")
# a.magnitude() 计算的是向量 a 的长度（模）。  模就像尺子量出来的长短，只关心大小，不关心方向。
print(f"|a| = {a.magnitude():.4f}")
print(f"余弦相似度 = {a.cosine_similarity(b):.4f}")

"""
余弦相似度 = 点积 ÷ （a 的模 × b 的模）
    点积 = 32
    |a| = √14 ≈ 3.7417
    |b| = √(4²+5²+6²) = √77 ≈ 8.7750
    分母 ≈ 3.7417 × 8.7750 ≈ 32.83
    比值 ≈ 32 ÷ 32.83 ≈ 0.9746
结果是一个 -1 到 1 之间的数：
    1 → 方向完全相同
    0 → 垂直
    -1 → 方向完全相反
这里的 0.9746 非常接近 1，说明 a 和 b 的方向几乎一样（虽然长短不同）。
"""

a + b = Vector([5, 7, 9])
a · b = 32
|a| = 3.7417
余弦相似度 = 0.9746


积（也叫内积）的计算方法是：对应分量相乘，再全部加起来。
1×4 + 2×5 + 3×6 = 4 + 10 + 18 = 32
结果是一个数字（标量），不是向量。
点积有什么用？

判断两个向量的方向关系：
- 点积 > 0 → 方向大致相同
- 点积 = 0 → 垂直（正交）
- 点积 < 0 → 方向相反

它也是余弦相似度的分子部分。


**为什么需要知道这些？**
点积和余弦相似度在机器学习中很常用：
- 找相似的文章（推荐系统）
- 找相似的词（Word2Vec 嵌入）
- 计算两个句子是否意思相近

向量的模用于归一化，让不同长度的向量能公平比较方向。

## 矩阵

In [2]:
class Matrix:
    def __init__(self, rows):
        self.rows = [list(row) for row in rows]
        self.shape = (len(self.rows), len(self.rows[0]))

    def __matmul__(self, other):
        if isinstance(other, Vector):
            return Vector([
                sum(self.rows[i][j] * other.components[j] for j in range(self.shape[1]))
                for i in range(self.shape[0])
            ])
        rows = []
        for i in range(self.shape[0]):
            row = []
            for j in range(other.shape[1]):
                row.append(sum(
                    self.rows[i][k] * other.rows[k][j]
                    for k in range(self.shape[1])
                ))
            rows.append(row)
        return Matrix(rows)

    def transpose(self):
        return Matrix([
            [self.rows[j][i] for j in range(self.shape[0])]
            for i in range(self.shape[1])
        ])

    def __repr__(self):
        return f"Matrix({self.rows})"


rotation_90 = Matrix([[0, -1], [1, 0]])
point = Vector([3, 1])

rotated = rotation_90 @ point
print(f"原始: {point}")
print(f"旋转 90°: {rotated}")

原始: Vector([3, 1])
旋转 90°: Vector([-1, 3])


几何意义：
- 原始点 (3, 1) 位于第一象限（右上方）。
- 逆时针旋转 90 度后，它移到了第二象限 (-1, 3)（左上方）。
- 这直观地展示了矩阵如何移动空间中的点——旋转、缩放、错切等一切线性变换都能用矩阵表示。

在 AI / 线性代数学习中的价值：
- 理解矩阵就是变换：一个神经网络层的权重矩阵本质上就是这样的“旋转+缩放”操作，只是作用在高维空间。
- 理解矩阵‑向量乘法：它是神经网络前向传播的核心计算。
- 为后续学习投影、特征分解、SVD 等打下直观基础。

In [3]:
import random

random.seed(42)
weights = Matrix([[random.gauss(0, 0.1) for _ in range(3)] for _ in range(2)])
input_vector = Vector([1.0, 0.5, -0.3])

output = weights @ input_vector
print(f"输入 (3D): {input_vector}")
print(f"输出 (2D): {output}")
print("这就是神经网络层所做的——矩阵乘法。")

输入 (3D): Vector([1.0, 0.5, -0.3])
输出 (2D): Vector([-0.019714737127338927, 0.10873956075097069])
这就是神经网络层所做的——矩阵乘法。


In [4]:
def is_linearly_independent(vectors):
    n = len(vectors)
    dim = len(vectors[0].components)
    mat = Matrix([v.components[:] for v in vectors])
    rows = [row[:] for row in mat.rows]
    rank = 0
    for col in range(dim):
        pivot = None
        for row in range(rank, len(rows)):
            if abs(rows[row][col]) > 1e-10:
                pivot = row
                break
        if pivot is None:
            continue
        rows[rank], rows[pivot] = rows[pivot], rows[rank]
        scale = rows[rank][col]
        rows[rank] = [x / scale for x in rows[rank]]
        for row in range(len(rows)):
            if row != rank and abs(rows[row][col]) > 1e-10:
                factor = rows[row][col]
                rows[row] = [rows[row][j] - factor * rows[rank][j] for j in range(dim)]
        rank += 1
    return rank == n


def project(a, b):
    scalar = a.dot(b) / b.dot(b)
    return Vector([scalar * x for x in b.components])


def gram_schmidt(vectors):
    orthonormal = []
    for v in vectors:
        w = v
        for u in orthonormal:
            proj = project(w, u)
            w = w - proj
        if w.magnitude() < 1e-10:
            continue
        orthonormal.append(w.normalize())
    return orthonormal


v1 = Vector([1, 0, 0])
v2 = Vector([1, 1, 0])
v3 = Vector([1, 1, 1])
basis = gram_schmidt([v1, v2, v3])
for i, u in enumerate(basis):
    print(f"u{i+1} = {u}")
    print(f"  |u{i+1}| = {u.magnitude():.6f}")

print(f"u1 · u2 = {basis[0].dot(basis[1]):.6f}")
print(f"u1 · u3 = {basis[0].dot(basis[2]):.6f}")
print(f"u2 · u3 = {basis[1].dot(basis[2]):.6f}")

u1 = Vector([1.0, 0.0, 0.0])
  |u1| = 1.000000
u2 = Vector([0.0, 1.0, 0.0])
  |u2| = 1.000000
u3 = Vector([0.0, 0.0, 1.0])
  |u3| = 1.000000
u1 · u2 = 0.000000
u1 · u3 = 0.000000
u2 · u3 = 0.000000
